# 15 build cnv consensus

**Purpose:** Combine CopyKAT, infercnvpy, and marker evidence.

**Original analysis notebook:** `15_final_CopyKAT_marker_CNV_consensus_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 34 — Final CopyKAT / infercnvpy / marker-based consensus annotation

Purpose:
- integrate the full microbatch CopyKAT result with marker-based annotation and previous infercnvpy interpretation
- define cluster-level and cell-level malignancy/CNV interpretation
- keep tumor-like labels scientifically cautious: **CopyKAT support ≠ automatic malignant proof for every cell**
- export TSV reports and optionally write a final annotated `.h5ad`

This notebook assumes notebook `33_integrate_CopyKAT_microbatch_predictions_HVG8000_r0_6.ipynb` has already run.

In [ ]:
# =========================
# SETTINGS
# =========================
from pathlib import Path
import os, gc, json, warnings
import numpy as np
import pandas as pd

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
FIG_DIR = PROJECT_DIR / "figures" / "copykat_final_consensus_HVG8000_r0_6"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TAG = "HVG8000_r0_6"

COPYKAT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad"
FALLBACK_H5AD = PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad"

FINAL_H5AD = PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad"

COPYKAT_FINAL_CELL_CALLS = METADATA_DIR / f"copykat_microbatch_final_cell_calls_{TAG}.tsv"
COPYKAT_CLUSTER_INTERPRETATION = METADATA_DIR / f"copykat_microbatch_cluster_interpretation_{TAG}.tsv"
COPYKAT_CLUSTER_SUMMARY = METADATA_DIR / f"copykat_microbatch_cluster_summary_{TAG}.tsv"
COPYKAT_OVERALL_SUMMARY = METADATA_DIR / f"copykat_microbatch_overall_summary_{TAG}.tsv"

INFERCNVPY_CONSENSUS = METADATA_DIR / f"infercnvpy_consensus_cluster_calls_{TAG}.tsv"

WRITE_FINAL_H5AD = True   # set False if disk/RAM is tight
MAKE_FIGURES = True
RANDOM_SEED = 42

CLUSTER_KEY_CANDIDATES = ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6", "leiden_scvi_0.6", "leiden_scvi_0_6", "leiden"]
ANNOTATION_COL_CANDIDATES = [
    "annotation_marker_based_HVG8000_r0_6",
    "annotation_marker_based",
    "annotation",
    "cell_type_marker_based",
]
COPYKAT_PRED_CANDIDATES = [
    "copykat_microbatch_pred",
    "copykat.pred",
    "copykat_pred",
    "copykat_prediction",
    "copykat_final_pred",
]

print("Project:", PROJECT_DIR)
print("CopyKAT h5ad exists:", COPYKAT_H5AD.exists(), COPYKAT_H5AD)
print("Final cell calls exists:", COPYKAT_FINAL_CELL_CALLS.exists(), COPYKAT_FINAL_CELL_CALLS)

In [ ]:
# =========================
# HELPERS
# =========================
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, sep="\t", index=False)
    os.replace(tmp, path)
    print(f"[SAVED] {path} shape={df.shape}")
    return path

def choose_first_existing_column(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"No {label} found. Tried: {candidates}. Available: {list(df.columns)[:80]}")
    return None

def safe_numeric_cluster(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    try:
        if s.endswith(".0"):
            return int(float(s))
        return int(s)
    except Exception:
        return s

def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.h5ad")
    if tmp.exists():
        tmp.unlink()
    adata_obj.write_h5ad(tmp)
    os.replace(tmp, path)
    print(f"[SAVED H5AD] {path}")

def normalize_copykat_pred(x):
    if pd.isna(x):
        return "missing"
    s = str(x).strip().lower()
    if s in {"aneuploid", "aneu", "tumor", "malignant"}:
        return "aneuploid"
    if s in {"diploid", "normal"}:
        return "diploid"
    if s in {"not.defined", "not_defined", "undefined", "na", "nan"}:
        return "not_defined"
    return s

In [ ]:
# =========================
# READ SUMMARY TABLES
# =========================
overall = pd.read_csv(COPYKAT_OVERALL_SUMMARY, sep="\t") if COPYKAT_OVERALL_SUMMARY.exists() else pd.DataFrame()
cluster_summary = pd.read_csv(COPYKAT_CLUSTER_SUMMARY, sep="\t")
cluster_interpretation = pd.read_csv(COPYKAT_CLUSTER_INTERPRETATION, sep="\t")

print("overall")
display(overall)
print("cluster interpretation")
display(cluster_interpretation)

In [ ]:
# =========================
# READ H5AD + ENSURE COPYKAT CELL CALLS ARE PRESENT
# =========================
import scanpy as sc

input_h5ad = COPYKAT_H5AD if COPYKAT_H5AD.exists() else FALLBACK_H5AD
adata = sc.read_h5ad(input_h5ad)
print(adata)

cluster_col = choose_first_existing_column(adata.obs, CLUSTER_KEY_CANDIDATES, label="cluster column")
annotation_col = choose_first_existing_column(adata.obs, ANNOTATION_COL_CANDIDATES, label="annotation column")
copykat_col = choose_first_existing_column(adata.obs, COPYKAT_PRED_CANDIDATES, required=False, label="CopyKAT prediction column")

print("cluster_col:", cluster_col)
print("annotation_col:", annotation_col)
print("copykat_col in adata:", copykat_col)

# If CopyKAT calls are not in adata.obs, merge final cell calls TSV.
if copykat_col is None:
    if not COPYKAT_FINAL_CELL_CALLS.exists():
        raise FileNotFoundError(COPYKAT_FINAL_CELL_CALLS)
    calls = pd.read_csv(COPYKAT_FINAL_CELL_CALLS, sep="\t")
    print("final cell calls columns:", list(calls.columns))
    cell_col = choose_first_existing_column(calls, ["cell_id", "cell.names", "cell", "barcode", "obs_name"], label="cell id in calls")
    pred_col = choose_first_existing_column(calls, ["copykat.pred", "copykat_pred", "copykat_prediction", "copykat_microbatch_pred"], label="prediction in calls")
    calls = calls[[cell_col, pred_col]].drop_duplicates(cell_col, keep="first")
    calls = calls.set_index(cell_col)
    adata.obs["copykat_microbatch_pred"] = adata.obs_names.map(calls[pred_col]).astype("object")
    copykat_col = "copykat_microbatch_pred"

adata.obs["copykat_pred_clean"] = adata.obs[copykat_col].map(normalize_copykat_pred).astype("category")
adata.obs["copykat_is_defined"] = adata.obs["copykat_pred_clean"].isin(["aneuploid", "diploid"])
adata.obs["copykat_is_aneuploid"] = adata.obs["copykat_pred_clean"].eq("aneuploid")

print(adata.obs["copykat_pred_clean"].value_counts(dropna=False))

In [ ]:
# =========================
# BUILD ROBUST CLUSTER-LEVEL CONSENSUS TABLE
# =========================

ci = cluster_interpretation.copy()
ci["cluster"] = ci["leiden_scvi_main"].map(safe_numeric_cluster).astype("object")
ci["annotation"] = ci["annotation_marker_based_HVG8000_r0_6"].astype("object")

def classify_cluster(row):
    ann = str(row.get("annotation", "")).lower()
    frac_defined = row.get("aneuploid_fraction_defined", np.nan)
    frac_all = row.get("aneuploid_fraction_all", np.nan)
    n_defined = row.get("n_defined", 0)
    n_cells = row.get("n_cells", 0)
    cluster = row.get("cluster", None)

    if n_defined < 50:
        return "insufficient_cells_review"

    is_marker_tumor = ann.startswith("putative_tumor")
    is_cycling = "cycling" in ann
    is_reference_like = ("oligodendrocyte" in ann) or ("endothelial" in ann)
    is_clear_immune = any(x in ann for x in ["myeloid", "t_nk", "b_plasma", "mast"])
    is_ambiguous = any(x in ann for x in ["ambiguous", "erythroid", "hb_high", "mes_ecm", "stromal"])

    if is_marker_tumor and frac_defined >= 0.65:
        return "malignant_like_copykat_supported"
    if is_cycling and frac_defined >= 0.65:
        return "cycling_malignant_like_candidate"
    if is_marker_tumor and frac_defined >= 0.50:
        return "malignant_like_copykat_moderate_support"
    if is_reference_like and frac_defined <= 0.25:
        return "non_malignant_reference_like"
    if is_clear_immune and frac_defined >= 0.60:
        return "marker_copykat_discordant_high_aneuploid_review"
    if is_ambiguous and frac_defined >= 0.60:
        return "ambiguous_high_aneuploid_review"
    if frac_defined >= 0.50:
        return "mixed_moderate_aneuploid_review"
    if frac_defined < 0.35:
        return "copykat_low_or_diploid_leaning"
    return "mixed_review"

ci["final_cnv_consensus_call"] = ci.apply(classify_cluster, axis=1)

# Add whether infercnvpy agreed if available.
if INFERCNVPY_CONSENSUS.exists():
    inf = pd.read_csv(INFERCNVPY_CONSENSUS, sep="\t")
    # robustly identify cluster column in infercnvpy table
    inf_cluster_col = choose_first_existing_column(inf, ["cluster", "leiden_scvi_main", "leiden_scvi_HVG8000_r0_6", "leiden_scvi_0.6"], required=False, label="infercnv cluster")
    if inf_cluster_col is not None:
        inf["cluster"] = inf[inf_cluster_col].map(safe_numeric_cluster).astype("object")
        keep_cols = ["cluster"] + [c for c in inf.columns if "cnv" in c.lower() or "call" in c.lower() or "consensus" in c.lower()]
        keep_cols = list(dict.fromkeys(keep_cols))
        ci = ci.merge(inf[keep_cols].drop_duplicates("cluster"), on="cluster", how="left", suffixes=("", "_infercnvpy"))
else:
    ci["infercnvpy_note"] = "infercnvpy consensus file not found"

# Make cluster labels as str for mapping to adata.obs
ci["cluster_str"] = ci["cluster"].astype(str).str.replace(r"\.0$", "", regex=True)
display(ci)

FINAL_CLUSTER_CONSENSUS_TSV = METADATA_DIR / f"final_copykat_marker_infercnvpy_cluster_consensus_{TAG}.tsv"
write_tsv_replace(ci, FINAL_CLUSTER_CONSENSUS_TSV)

In [ ]:
# =========================
# MAP CLUSTER CONSENSUS BACK TO CELLS
# =========================
cluster_to_call = dict(zip(ci["cluster_str"], ci["final_cnv_consensus_call"]))
cluster_to_frac = dict(zip(ci["cluster_str"], ci["aneuploid_fraction_defined"]))
cluster_to_ann = dict(zip(ci["cluster_str"], ci["annotation"]))

obs_cluster_str = adata.obs[cluster_col].astype(str).str.replace(r"\.0$", "", regex=True)

adata.obs["final_cnv_consensus_by_cluster"] = obs_cluster_str.map(cluster_to_call).fillna("unmapped_cluster").astype("category")
adata.obs["copykat_cluster_aneuploid_fraction_defined"] = obs_cluster_str.map(cluster_to_frac).astype(float)
adata.obs["final_marker_annotation_for_cnv"] = obs_cluster_str.map(cluster_to_ann).fillna(adata.obs[annotation_col].astype(str)).astype("category")

def final_malignancy_from_consensus(x):
    x = str(x)
    if x in ["malignant_like_copykat_supported", "malignant_like_copykat_moderate_support"]:
        return "malignant_like"
    if x == "cycling_malignant_like_candidate":
        return "cycling_malignant_like_candidate"
    if x in ["non_malignant_reference_like", "copykat_low_or_diploid_leaning"]:
        return "non_malignant_or_diploid_leaning"
    if x == "unmapped_cluster":
        return "unmapped"
    return "review"

adata.obs["final_malignancy_group_copykat_marker"] = adata.obs["final_cnv_consensus_by_cluster"].map(final_malignancy_from_consensus).astype("category")

# Optional: direct cell-level CopyKAT group, separate from cluster-level final interpretation.
def cell_level_group(pred):
    pred = normalize_copykat_pred(pred)
    if pred == "aneuploid": return "copykat_cell_aneuploid"
    if pred == "diploid": return "copykat_cell_diploid"
    if pred == "not_defined": return "copykat_cell_not_defined"
    return "copykat_cell_missing_or_other"

adata.obs["copykat_cell_level_group"] = adata.obs["copykat_pred_clean"].map(cell_level_group).astype("category")

print(adata.obs["final_malignancy_group_copykat_marker"].value_counts())
print(adata.obs["final_cnv_consensus_by_cluster"].value_counts())

In [ ]:
# =========================
# EXPORT SUMMARY TABLES
# =========================
FINAL_CELL_COUNTS_TSV = METADATA_DIR / f"final_copykat_marker_malignancy_group_counts_{TAG}.tsv"
FINAL_CLUSTER_COUNTS_TSV = METADATA_DIR / f"final_copykat_marker_cluster_counts_{TAG}.tsv"
FINAL_ANNOTATION_COUNTS_TSV = METADATA_DIR / f"final_copykat_marker_annotation_counts_{TAG}.tsv"

cell_counts = (
    adata.obs.groupby(["final_malignancy_group_copykat_marker", "copykat_pred_clean"], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values(["final_malignancy_group_copykat_marker", "copykat_pred_clean"])
)
write_tsv_replace(cell_counts, FINAL_CELL_COUNTS_TSV)

cluster_counts = (
    adata.obs.assign(cluster=obs_cluster_str)
    .groupby(["cluster", annotation_col, "final_cnv_consensus_by_cluster", "final_malignancy_group_copykat_marker"], observed=True)
    .agg(
        n_cells=("copykat_pred_clean", "size"),
        n_copykat_aneuploid=("copykat_pred_clean", lambda s: (s == "aneuploid").sum()),
        n_copykat_diploid=("copykat_pred_clean", lambda s: (s == "diploid").sum()),
        n_copykat_not_defined=("copykat_pred_clean", lambda s: (s == "not_defined").sum()),
        n_copykat_missing=("copykat_pred_clean", lambda s: (s == "missing").sum()),
    )
    .reset_index()
)
cluster_counts["n_defined"] = cluster_counts["n_copykat_aneuploid"] + cluster_counts["n_copykat_diploid"]
cluster_counts["aneuploid_fraction_defined"] = cluster_counts["n_copykat_aneuploid"] / cluster_counts["n_defined"].replace(0, np.nan)
cluster_counts["aneuploid_fraction_all"] = cluster_counts["n_copykat_aneuploid"] / cluster_counts["n_cells"].replace(0, np.nan)
cluster_counts = cluster_counts.sort_values(["aneuploid_fraction_defined", "n_cells"], ascending=[False, False])
write_tsv_replace(cluster_counts, FINAL_CLUSTER_COUNTS_TSV)

annotation_counts = (
    adata.obs.groupby([annotation_col, "final_malignancy_group_copykat_marker"], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values([annotation_col, "n_cells"], ascending=[True, False])
)
write_tsv_replace(annotation_counts, FINAL_ANNOTATION_COUNTS_TSV)

display(cluster_counts)
display(annotation_counts)

In [ ]:
# =========================
# MAKE FIGURES
# =========================
if MAKE_FIGURES:
    import matplotlib.pyplot as plt
    import scanpy as sc

    if "X_umap" in adata.obsm:
        for color in [
            "copykat_pred_clean",
            "final_cnv_consensus_by_cluster",
            "final_malignancy_group_copykat_marker",
            annotation_col,
            cluster_col,
        ]:
            try:
                sc.pl.umap(
                    adata,
                    color=color,
                    frameon=False,
                    size=2,
                    legend_loc="right margin",
                    show=False,
                )
                out = FIG_DIR / f"umap_{color}_{TAG}.png"
                plt.savefig(out, dpi=220, bbox_inches="tight")
                plt.close()
                print("[SAVED FIG]", out)
            except Exception as e:
                print("[WARNING] could not plot", color, repr(e))
    else:
        print("[SKIP] No X_umap in adata.obsm")

In [ ]:
# =========================
# OUTPUT MANIFEST + OPTIONAL H5AD
# =========================
outputs = [
    ("final_cluster_consensus", FINAL_CLUSTER_CONSENSUS_TSV),
    ("final_cell_counts", FINAL_CELL_COUNTS_TSV),
    ("final_cluster_counts", FINAL_CLUSTER_COUNTS_TSV),
    ("final_annotation_counts", FINAL_ANNOTATION_COUNTS_TSV),
]

if WRITE_FINAL_H5AD:
    write_h5ad_replace(adata, FINAL_H5AD)
    outputs.append(("final_h5ad", FINAL_H5AD))
else:
    print("[SKIP] WRITE_FINAL_H5AD=False")

manifest = pd.DataFrame([
    {
        "output_label": label,
        "path": str(path),
        "exists": Path(path).exists(),
        "size_mb": round(Path(path).stat().st_size / 1024**2, 3) if Path(path).exists() else np.nan,
    }
    for label, path in outputs
])

MANIFEST_TSV = METADATA_DIR / f"final_copykat_marker_consensus_output_manifest_{TAG}.tsv"
write_tsv_replace(manifest, MANIFEST_TSV)
display(manifest)

print("[DONE] final CopyKAT/marker consensus notebook")